# Exercise 2.2: CorrDiff Regional Downscaling

**Level**: Intermediate | **Time**: 30 min | **GPU**: Required

Understand CorrDiff — NVIDIA's generative diffusion model that takes a coarse 25km global forecast and produces 2-3km local detail. 22x faster and 1,300x more energy efficient than traditional dynamical downscaling.

---

**Two-stage architecture:**
1. **Regression UNet** → predicts the conditional mean (smooth baseline)
2. **Diffusion Model** → predicts residual corrections (fine-scale detail + uncertainty)

Each run produces a different plausible realization — this IS the uncertainty quantification.

In [ ]:
# Install Earth2Studio + dependencies (don't install torch — Colab has it)
!pip install -q "earth2studio>=0.13.0" matplotlib scipy numpy

## Part 1: Understand the Architecture (Synthetic Demo)

This demo illustrates CorrDiff's two-stage approach without needing the actual NIM container. The concepts are identical.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom

np.random.seed(42)

# Synthetic coarse field (~25km, 40×40 grid)
n_coarse = 40
x_c, y_c = np.meshgrid(np.linspace(0, 10, n_coarse), np.linspace(0, 10, n_coarse))
coarse = np.sin(x_c) * np.cos(y_c) * 10 + np.random.normal(0, 0.5, (n_coarse, n_coarse))

# Synthetic fine field (~3km, 320×320 grid — 8x upscale)
scale = 8
n_fine = n_coarse * scale
x_f, y_f = np.meshgrid(np.linspace(0, 10, n_fine), np.linspace(0, 10, n_fine))

# STAGE 1: Regression UNet — smooth upscaling (conditional mean)
regression = zoom(coarse, scale, order=3)

# STAGE 2: Diffusion — generate 4 stochastic samples
samples = []
for i in range(4):
    fine_detail = np.sin(x_f * 5) * np.cos(y_f * 7) * 0.3 + np.random.normal(0, 0.1, (n_fine, n_fine))
    samples.append(regression + fine_detail)

print(f"Coarse: {coarse.shape} → Fine: {samples[0].shape} ({scale}x upscale)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: The two-stage architecture
axes[0,0].contourf(x_c, y_c, coarse, levels=30, cmap='viridis')
axes[0,0].set_title('INPUT: Coarse (~25km)', fontsize=13)

axes[0,1].contourf(x_f, y_f, regression, levels=30, cmap='viridis')
axes[0,1].set_title('STAGE 1: Regression UNet\n(smooth upscaling)', fontsize=13)

residual = samples[0] - regression
axes[0,2].contourf(x_f, y_f, residual, levels=30, cmap='coolwarm')
axes[0,2].set_title('STAGE 2: Diffusion Residual\n(fine-scale corrections)', fontsize=13)

# Row 2: Multiple stochastic samples
for i in range(3):
    axes[1,i].contourf(x_f, y_f, samples[i], levels=30, cmap='viridis')
    axes[1,i].set_title(f'OUTPUT: Sample {i+1} (~3km)\n(regression + diffusion)', fontsize=13)

fig.suptitle('CorrDiff: Two-Stage Downscaling Architecture', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Part 2: Inter-Sample Variability = Uncertainty

Where samples differ most → highest uncertainty in fine-scale outcome.

In [ ]:
sample_std = np.std(samples, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cf1 = axes[0].contourf(x_f, y_f, samples[0], levels=30, cmap='viridis')
axes[0].set_title('Single Realization')
plt.colorbar(cf1, ax=axes[0])

cf2 = axes[1].contourf(x_f, y_f, sample_std, levels=30, cmap='YlOrRd')
axes[1].set_title('Inter-Sample Std Dev\n(High = uncertain fine-scale outcome)')
plt.colorbar(cf2, ax=axes[1], label='Std Dev')

fig.suptitle('CorrDiff Uncertainty from Stochastic Sampling', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 3: Power Spectrum — Resolution Matters

Does downscaling actually ADD information, or just interpolate? Check the power spectrum.

In [ ]:
def radial_power_spectrum(field):
    """Compute radially-averaged 2D power spectrum."""
    fft = np.fft.fft2(field)
    power = np.abs(fft)**2
    ny, nx = field.shape
    ky = np.fft.fftfreq(ny)[:, None]
    kx = np.fft.fftfreq(nx)[None, :]
    kr = np.sqrt(kx**2 + ky**2)
    
    k_bins = np.arange(0, 0.5, 0.005)
    spectrum = np.zeros(len(k_bins) - 1)
    for i in range(len(k_bins) - 1):
        mask = (kr >= k_bins[i]) & (kr < k_bins[i+1])
        if mask.any():
            spectrum[i] = np.mean(power[mask])
    return 0.5 * (k_bins[:-1] + k_bins[1:]), spectrum

# Compare spectra
k_reg, spec_reg = radial_power_spectrum(regression)
k_diff, spec_diff = radial_power_spectrum(samples[0])

# Upscale coarse to same grid for comparison
coarse_upscaled = zoom(coarse, scale, order=1)  # Bilinear = no new info
k_bilin, spec_bilin = radial_power_spectrum(coarse_upscaled)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(k_bilin, spec_bilin, 'g-', linewidth=2, label='Bilinear interpolation (no new info)', alpha=0.7)
ax.loglog(k_reg, spec_reg, 'b-', linewidth=2, label='Stage 1: Regression UNet')
ax.loglog(k_diff, spec_diff, 'r-', linewidth=2, label='Stage 1+2: Full CorrDiff')
ax.set_xlabel('Wavenumber')
ax.set_ylabel('Power')
ax.set_title('Power Spectrum: Does Downscaling Add Real Information?\n'
             'CorrDiff\'s diffusion stage adds realistic high-frequency content')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Running CorrDiff for Real (NIM Container)

If you have Docker + NVIDIA GPU + NGC API key, you can run the actual model:

```bash
# In terminal (not Colab)
export NGC_API_KEY="your-key"
docker pull nvcr.io/nim/nvidia/corrdiff:1.1.0
docker run --rm --runtime=nvidia --gpus all --shm-size 4g \
    -p 8000:8000 -e NGC_API_KEY=$NGC_API_KEY \
    nvcr.io/nim/nvidia/corrdiff:1.1.0
```

Then call the API:

In [ ]:
# This cell is for reference — only works with NIM running locally

nim_example = """
import requests

# Health check
resp = requests.get("http://localhost:8000/v1/health/ready")
print(resp.json())  # {"status": "ready"}

# Run inference
files = {"input_array": ("input.npy", open("corrdiff_input.npy", "rb"))}
params = {"samples": 4, "steps": 14, "seed": 42}
response = requests.post("http://localhost:8000/v1/infer", data=params, files=files)

# Response is a .tar of numpy arrays — one per stochastic sample
"""
print(nim_example)

## Part 5: End-to-End Pipeline Concept

The production architecture chains global → local:

```
GFS data → FCN3 (global 25km) → CorrDiff (regional 3km) → StormScope (1km nowcast)
```

In [ ]:
# Demonstrate the pipeline concept
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Stage 0: Raw observations (simulated)
obs = np.random.normal(0, 1, (10, 10))
axes[0].imshow(obs, cmap='viridis', aspect='auto')
axes[0].set_title('Observations\n(satellites, stations)', fontsize=11)
axes[0].set_xticks([]); axes[0].set_yticks([])

# Stage 1: Global forecast (coarse)
global_fcst = zoom(np.sin(x_c[:20,:20]) * 5, 2, order=3)
axes[1].imshow(global_fcst, cmap='RdYlBu_r', aspect='auto')
axes[1].set_title('FCN3 Global Forecast\n(~25km, 0-15 days)', fontsize=11)
axes[1].set_xticks([]); axes[1].set_yticks([])

# Stage 2: Downscaled
downscaled = zoom(global_fcst, 4, order=3) + np.random.normal(0, 0.3, (160, 160))
axes[2].imshow(downscaled, cmap='RdYlBu_r', aspect='auto')
axes[2].set_title('CorrDiff Downscaled\n(~3km, regional)', fontsize=11)
axes[2].set_xticks([]); axes[2].set_yticks([])

# Stage 3: Nowcast
nowcast = zoom(downscaled[:80,:80], 2, order=3) + np.random.normal(0, 0.5, (160, 160))
axes[3].imshow(nowcast, cmap='RdYlBu_r', aspect='auto')
axes[3].set_title('StormScope Nowcast\n(~1km, 0-6 hours)', fontsize=11)
axes[3].set_xticks([]); axes[3].set_yticks([])

# Arrows
for i in range(3):
    fig.text(0.26 + i*0.235, 0.5, '→', fontsize=30, ha='center', va='center',
             transform=fig.transFigure, fontweight='bold', color='gray')

fig.suptitle('NVIDIA Earth-2 Full Pipeline: Global → Regional → Storm-Scale',
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

---

## Exercises

1. In the synthetic demo, increase the number of samples from 4 to 20. Does the std map get smoother?
2. Compare the power spectra more carefully — at what wavenumber does bilinear interpolation "run out" of information?
3. If you have Docker + GPU, pull the CorrDiff NIM and run end-to-end
4. **Challenge**: Chain a real FCN3 forecast (from Exercise 1.1) into CorrDiff input

**Key takeaway**: CorrDiff's diffusion stage doesn't just smooth-upscale — it generates physically plausible fine-scale structure that differs between samples. This is generative AI applied to physics.